# Jira API Direct Fetch

Fetches Jira issue data directly from the Jira Cloud API, replacing the manual CSV export workflow.

**Prerequisites — Generate an API Token:**
1. Log in to https://id.atlassian.com/manage-profile/security/api-tokens
2. Click **Create API token** and copy the token
3. Paste it into the `API_TOKEN` variable below

In [ ]:
import requests
import pandas as pd
import json
import os
import re
from datetime import datetime, timezone
import pytz
from requests.auth import HTTPBasicAuth

In [ ]:
# ── Configuration ─────────────────────────────────────────
JIRA_URL   = "https://westfund.atlassian.net"
EMAIL      = "zengsh@westfund.com.au"       # ← your Jira login email
API_TOKEN  = "ATATT3xFfGF0Fa0-6EpN0S3xJPkI5Ia6HBgzXK3szQQ6pv9QTgneVaSeL3K4ex4pRkKeVWVlBu02hhk7jq3W3bG2bsQfxI3dpvueRo5kRP9pWJWH6uok67u6slJqkj2ilaBEQUZ5xkMwCms02uNq5wosR6b1Zii4StRyJS2DasTvRgPdWeehoOc=0683F66A"          # ← your API Token

# ── Date filter (only fetch tickets created on or after this date) ──
START_DATE = "2026-01-01"   # ← default, change as needed
# ──────────────────────────────────────────────────────────

# Output path
OUTPUT_DIR = r"\\prdeqs01\QlikData\Jira_\output"

# Sydney timezone
SYDNEY_TZ = pytz.timezone("Australia/Sydney")

# JQL is built automatically — no need to edit this line
JQL = f"project = DIG AND created >= '{START_DATE}' ORDER BY created DESC"
print(f"JQL: {JQL}")

AUTH    = HTTPBasicAuth(EMAIL, API_TOKEN)
HEADERS = {"Accept": "application/json"}

JQL: project = DIG AND created >= '2026-01-01' ORDER BY created DESC


In [ ]:
def fetch_issue_ids(jql: str, batch_size: int = 100) -> list:
    """Fetch issue IDs using POST /search/jql with cursor-based pagination."""
    url = f"{JIRA_URL}/rest/api/3/search/jql"
    post_headers = {**HEADERS, "Content-Type": "application/json"}
    all_issues = []
    next_page_token = None

    while True:
        body = {"jql": jql, "maxResults": batch_size}
        if next_page_token:
            body["nextPageToken"] = next_page_token

        resp = requests.post(url, headers=post_headers, auth=AUTH, json=body)
        if not resp.ok:
            print(f"HTTP {resp.status_code}: {resp.text[:800]}")
            resp.raise_for_status()

        data = resp.json()
        issues = data.get("issues", [])
        all_issues.extend(issues)

        is_last = data.get("isLast", True)
        next_page_token = data.get("nextPageToken")
        print(f"  Fetched {len(all_issues)} | isLast={is_last}")

        if is_last or not issues or not next_page_token:
            break

    return all_issues


def fetch_issue_detail(issue_id: str) -> dict:
    """Fetch full issue detail (including changelog) by issue ID."""
    url = f"{JIRA_URL}/rest/api/3/issue/{issue_id}"
    resp = requests.get(url, headers=HEADERS, auth=AUTH,
                        params={"expand": "changelog",
                                "fields": "summary,status,customfield_10187,reporter,assignee,customfield_10148,created"})
    resp.raise_for_status()
    return resp.json()


print("Step 1: Search issues...")
raw_issues = fetch_issue_ids(JQL)
print(f"\nFound {len(raw_issues)} issues")

print("\nStep 2: Fetch detail + changelog for each issue...")
issues = []
for i, raw in enumerate(raw_issues):
    issue_id = raw["id"]
    detail = fetch_issue_detail(issue_id)
    issues.append(detail)
    if (i + 1) % 100 == 0:
        print(f"  Processed {i + 1}/{len(raw_issues)}")

print(f"\nDone — {len(issues)} issues fetched")
if issues:
    print("Sample issue key:", issues[0].get("key"))

Step 1: Search issues...
  Fetched 100 | isLast=False
  Fetched 158 | isLast=True

Found 158 issues

Step 2: Fetch detail + changelog for each issue...
  Processed 100/158

Done — 158 issues fetched
Sample issue key: DIG-1951


In [ ]:
def seconds_to_duration(seconds: float) -> str:
    """Convert seconds to a human-readable duration string e.g. '3d 2h 15m 10s'."""
    seconds = int(seconds)
    d, rem = divmod(seconds, 86400)
    h, rem = divmod(rem, 3600)
    m, s   = divmod(rem, 60)
    parts = []
    if d: parts.append(f"{d}d")
    if h: parts.append(f"{h}h")
    if m: parts.append(f"{m}m")
    if s: parts.append(f"{s}s")
    return " ".join(parts) if parts else "0s"


def parse_changelog(issue: dict) -> list:
    """Extract status change history from issue changelog.
    Each row contains: time_point, to, statusCategory, duration, duration_cal.
    duration = time spent in that status before the next transition.
    """
    histories = issue.get("changelog", {}).get("histories", [])
    status_changes = []
    for h in histories:
        for item in h.get("items", []):
            if item.get("field") == "status":
                status_changes.append({
                    "created":        h["created"],
                    "to":             item.get("toString", ""),
                    "statusCategory": item.get("to", ""),
                })
    status_changes.sort(key=lambda x: x["created"])

    rows = []
    for i, sc in enumerate(status_changes):
        ts = datetime.fromisoformat(sc["created"].replace("Z", "+00:00"))
        ts_sydney = ts.astimezone(SYDNEY_TZ)
        if i + 1 < len(status_changes):
            ts_next = datetime.fromisoformat(status_changes[i+1]["created"].replace("Z", "+00:00"))
            duration_sec = (ts_next - ts).total_seconds()
        else:
            # Last status: measure time elapsed since the transition until now
            duration_sec = (datetime.now(timezone.utc) - ts).total_seconds()
        rows.append({
            "time_point":     ts_sydney.replace(tzinfo=None),
            "to":             sc["to"],
            "statusCategory": sc["statusCategory"],
            "duration":       seconds_to_duration(duration_sec),
            "duration_cal":   int(duration_sec),
        })
    return rows


def get_display_name(field_val) -> str:
    """Extract displayName from a user field object."""
    if isinstance(field_val, dict):
        return field_val.get("displayName", "")
    return ""


def get_option_value(field_val) -> str:
    """Extract value from a single-select or multi-select custom field."""
    if isinstance(field_val, list):
        return ", ".join(v.get("value", "") for v in field_val if isinstance(v, dict))
    if isinstance(field_val, dict):
        return field_val.get("value", "")
    return field_val or ""


print("Parsing changelog...")
all_rows = []

for issue in issues:
    fields  = issue.get("fields", {})
    key     = issue["key"]
    summary = fields.get("summary", "")
    status  = fields.get("status", {}).get("name", "")

    business_unit = get_option_value(fields.get("customfield_10187"))
    reporter      = get_display_name(fields.get("reporter"))
    assignee      = get_display_name(fields.get("assignee"))
    required_by   = fields.get("customfield_10148", "")  # Required By date

    # Convert created timestamp to Sydney time (timezone-naive)
    created_raw = fields.get("created")
    if created_raw:
        created_dt = datetime.fromisoformat(created_raw.replace("Z", "+00:00")).astimezone(SYDNEY_TZ)
        created = created_dt.replace(tzinfo=None)
    else:
        created = None

    issue_label  = f"({key}) {summary}"
    history_rows = parse_changelog(issue)

    base = {
        "business_unit": business_unit,
        "issue_number":  key,
        "issue":         issue_label,
        "status":        status,
        "created":       created,
        "reporter":      reporter,
        "assignee":      assignee,
        "required_by":   required_by,
    }

    if not history_rows:
        # No status transitions recorded — ticket has never changed status
        all_rows.append({**base, "index_in_issue": 1, "time_point": None,
                         "to": status, "statusCategory": "", "duration": None, "duration_cal": 0})
    else:
        for idx, row in enumerate(history_rows, start=1):
            all_rows.append({**base, "index_in_issue": idx, **row})

df = pd.DataFrame(all_rows)

# latest: 1 = most recent status transition, 2 = second most recent, etc.
# Tickets with no changelog (time_point = None) always get latest = 1
df['latest'] = (
    df.groupby('issue_number')['time_point']
    .rank(method='first', ascending=False, na_option='bottom')
    .astype(int)
)

print(f"Generated {len(df)} rows")
df

Parsing changelog...
Generated 475 rows


,business_unit,issue_number,issue,status,created,reporter,assignee,required_by,index_in_issue,time_point,to,statusCategory,duration,duration_cal,latest
0,,DIG-1951,(DIG-1951) Email preference - unsubscribed but...,Open,2026-04-07 10:46:21.345,Alexandra Blakely,Brendan Mercieca,,1,NaT,Open,,None,0,1
1,,DIG-1950,(DIG-1950) Web Chat enhancements - discovery a...,Waiting for approval,2026-04-07 08:39:40.472,Fiona Goldrick,Magdalena Herceg,,1,NaT,Waiting for approval,,None,0,1
2,,DIG-1949,(DIG-1949) Protecht #1208240 - promo code giv...,In Review,2026-04-02 08:32:12.523,John Iverach,Brendan Mercieca,,1,NaT,In Review,,None,0,1
3,,DIG-1948,"(DIG-1948) Update to Notification Type ""amboin...",Open,2026-04-01 17:52:10.443,Amy Spelleken,Brendan Mercieca,,1,2026-04-01 18:06:11.661,Open,1,5d 21h 19m 57s,508797,1
4,,DIG-1947,(DIG-1947) Potential member advised quoting to...,Closed,2026-04-01 11:07:56.899,Pam Riik,Brendan Mercieca,,1,2026-04-01 11:36:24.398,In Progress,3,3s,3,4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
470,,DIG-1785,(DIG-1785) Adding Assigned to RPA status to Ch...,Closed,2026-01-06 14:12:40.626,Ilia Vavilin,,,2,2026-01-13 10:15:37.927,Closed,6,84d 5h 10m 31s,7276231,1
471,,DIG-1784,(DIG-1784) 403 error code,Closed,2026-01-05 12:40:48.738,Tenika Strachan,Brendan Mercieca,,1,2026-01-19 13:24:02.756,In Progress,3,2s,2,4
472,,DIG-1784,(DIG-1784) 403 error code,Closed,2026-01-05 12:40:48.738,Tenika Strachan,Brendan Mercieca,,2,2026-01-19 13:24:05.565,Waiting for Response,10517,1h 26m 48s,5208,3
473,,DIG-1784,(DIG-1784) 403 error code,Closed,2026-01-05 12:40:48.738,Tenika Strachan,Brendan Mercieca,,3,2026-01-19 14:50:54.479,In Progress,3,6m 39s,399,2


In [ ]:
# Data overview
print("Issue count:", df['issue_number'].nunique())
print("\nStatus distribution:")
print(df.drop_duplicates('issue_number')['status'].value_counts())
print("\nColumn info:")
df.info()

# Check rows per ticket
rows_per_ticket = df.groupby('issue_number').size().value_counts().sort_index()
print("\nRows per ticket distribution:")
print(rows_per_ticket.rename_axis('row_count').reset_index(name='ticket_count'))

single_row_count = (df.groupby('issue_number').size() == 1).sum()
print(f"\nTickets with only 1 row (no status transitions): {single_row_count} / {df['issue_number'].nunique()}")

Issue count: 158

Status distribution:
status
Closed                  121
DEV Required             27
Open                      3
In Review                 2
Parked or Blocked         2
Waiting for approval      1
Cancelled                 1
In Progress               1
Name: count, dtype: int64

Column info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 475 entries, 0 to 474
Data columns (total 15 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   business_unit   475 non-null    object        
 1   issue_number    475 non-null    object        
 2   issue           475 non-null    object        
 3   status          475 non-null    object        
 4   created         475 non-null    datetime64[ns]
 5   reporter        475 non-null    object        
 6   assignee        475 non-null    object        
 7   required_by     475 non-null    object        
 8   index_in_issue  475 non-null    int64         
 9   time_poi

In [ ]:
# Save output
today_str = datetime.now(SYDNEY_TZ).strftime("%d%m%Y")
file_name = f"jira_{today_str}.csv"
file_path = os.path.join(OUTPUT_DIR, file_name)

os.makedirs(OUTPUT_DIR, exist_ok=True)
df.to_csv(file_path, index=False, encoding="utf-8-sig")

print(f"Saved to: {file_path}")
print(f"{len(df)} rows | {df['issue_number'].nunique()} issues")

Saved to: \\prdeqs01\QlikData\Jira_\output\jira_07042026.csv
475 rows | 158 issues
